# 01 - NIfTI, BOLD fMRI and T1w MRI basics

Real-data learning notebook for **OpenNeuro ds000117 (Wakeman & Henson)**.

Goals:
- download a minimal subset of the public dataset directly in Google Colab;
- load a real BOLD NIfTI with NiBabel;
- inspect shape, voxel size and TR;
- visualize orthogonal slices from one fMRI volume;
- load the participant's T1w structural MRI;
- compare fMRI and T1w spatial sampling;
- inspect their affine matrices.

Raw neuroimaging data are downloaded at runtime and are **not stored in this repository**.


## 1. Environment

The notebook is designed to run in Google Colab. `boto3` is used for anonymous access to the public OpenNeuro S3 bucket, and `nibabel` is used to read NIfTI files.


In [ ]:
!pip -q install nibabel nilearn boto3


In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import nibabel as nib
import matplotlib.pyplot as plt


## 2. Connect to the public OpenNeuro bucket

We first inspect the BIDS `func/` directory for `sub-01` rather than downloading the whole multimodal dataset.


In [ ]:
s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED),
)

bucket = "openneuro.org"
func_prefix = "ds000117/sub-01/ses-mri/func/"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=func_prefix,
)

for obj in response.get("Contents", []):
    print(obj["Key"])


## 3. Download and load one BOLD fMRI run

For the first practical checkpoint we only need `run-01` from the face-recognition task.


In [ ]:
bold_key = (
    "ds000117/sub-01/ses-mri/func/"
    "sub-01_ses-mri_task-facerecognition_run-01_bold.nii.gz"
)

bold_file = "/content/run01_bold.nii.gz"

s3.download_file(bucket, bold_key, bold_file)
print("Downloaded:", bold_file)


In [ ]:
func_img = nib.load(bold_file)

print("Shape:", func_img.shape)
print("Voxel sizes / TR:", func_img.header.get_zooms())


Expected values for this run:

```text
shape = (64, 64, 33, 208)
voxel sizes / TR = (3.0, 3.0, 3.75, 2.0)
```

Interpretation: one volume contains a `64 × 64 × 33` voxel grid, and the run contains 208 such volumes. A new volume is acquired every 2 seconds.


## 4. Extract and visualize one 3D fMRI volume

A 4D fMRI array has the conceptual form `X × Y × Z × T`. Selecting one time index removes the temporal dimension and leaves one 3D volume.


In [ ]:
func_data = func_img.get_fdata()
first_volume = func_data[:, :, :, 0]

print("First volume shape:", first_volume.shape)


In [ ]:
x = first_volume.shape[0] // 2
y = first_volume.shape[1] // 2
z = first_volume.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(first_volume[x, :, :].T, cmap="gray", origin="lower")
axes[0].set_title("Sagittal")

axes[1].imshow(first_volume[:, y, :].T, cmap="gray", origin="lower")
axes[1].set_title("Coronal")

axes[2].imshow(first_volume[:, :, z].T, cmap="gray", origin="lower")
axes[2].set_title("Axial")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 5. Download the participant's T1w structural MRI

The structural T1w image provides much finer anatomical sampling than the BOLD acquisition. We again inspect the BIDS directory first and select only the T1w NIfTI.


In [ ]:
anat_prefix = "ds000117/sub-01/ses-mri/anat/"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=anat_prefix,
)

for obj in response.get("Contents", []):
    print(obj["Key"])


In [ ]:
anat_keys = [
    obj["Key"]
    for obj in response.get("Contents", [])
]

t1_candidates = [
    key for key in anat_keys
    if key.endswith("_T1w.nii.gz")
]

print(t1_candidates)


In [ ]:
t1_key = t1_candidates[0]
t1_file = "/content/sub01_T1w.nii.gz"

s3.download_file(bucket, t1_key, t1_file)

anat_img = nib.load(t1_file)

print("T1w shape:", anat_img.shape)
print("T1w voxel size:", anat_img.header.get_zooms())


For `sub-01`, the T1w image has shape `(192, 256, 256)` with `1 × 1 × 1 mm` voxels.

This contrasts with the BOLD acquisition (`3 × 3 × 3.75 mm` voxels). One BOLD voxel therefore occupies about `33.75 mm³`, compared with `1 mm³` for one T1w voxel.


## 6. Compare central T1w and fMRI slices

This is a **visual comparison of central array slices**, not yet a spatial registration. The two images have different voxel grids, so equal or central voxel indices do not automatically represent exactly the same physical location.


In [ ]:
anat_data = anat_img.get_fdata()
func_volume = func_data[:, :, :, 0]

# Central indices in each image grid
ax = anat_data.shape[0] // 2
ay = anat_data.shape[1] // 2
az = anat_data.shape[2] // 2

fx = func_volume.shape[0] // 2
fy = func_volume.shape[1] // 2
fz = func_volume.shape[2] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# T1w
axes[0, 0].imshow(anat_data[ax, :, :].T, cmap="gray", origin="lower")
axes[0, 0].set_title("T1w - Sagittal")
axes[0, 1].imshow(anat_data[:, ay, :].T, cmap="gray", origin="lower")
axes[0, 1].set_title("T1w - Coronal")
axes[0, 2].imshow(anat_data[:, :, az].T, cmap="gray", origin="lower")
axes[0, 2].set_title("T1w - Axial")

# fMRI
axes[1, 0].imshow(func_volume[fx, :, :].T, cmap="gray", origin="lower")
axes[1, 0].set_title("fMRI - Sagittal")
axes[1, 1].imshow(func_volume[:, fy, :].T, cmap="gray", origin="lower")
axes[1, 1].set_title("fMRI - Coronal")
axes[1, 2].imshow(func_volume[:, :, fz].T, cmap="gray", origin="lower")
axes[1, 2].set_title("fMRI - Axial")

for ax_ in axes.ravel():
    ax_.axis("off")

plt.tight_layout()
plt.show()


## 7. Inspect affine matrices

Voxel coordinates such as `[30, 30, 15]` are addresses inside an array. NIfTI affine matrices map these voxel indices to physical/world coordinates in millimetres.

At this checkpoint we only inspect the real matrices. Applying the affine and cross-image spatial correspondence are the next learning step.


In [ ]:
print("fMRI affine:")
print(func_img.affine)

print()

print("T1w affine:")
print(anat_img.affine)


## Checkpoint

Completed in this notebook:
- loaded real OpenNeuro BOLD data;
- inspected 4D shape, voxel size and TR;
- extracted and visualized one 3D fMRI volume;
- loaded a real T1w anatomical image from the same participant;
- compared fMRI and T1w spatial resolution;
- inspected both affine matrices.

Next: `voxel index → affine → world coordinates → correspondence between fMRI and T1w space`.
